# Explore training_run results

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("./training_runs.csv", sep=",")

Comprobamos cuantos runs tenemos

In [4]:
print("Numero de runs : ",str(df.shape[0]))
print("Por caso de uso - experimento")
df.value_counts("experiment_id")

Numero de runs :  80
Por caso de uso - experimento


experiment_id
3    36
1    34
2    10
Name: count, dtype: int64

recogemos las columnas de metadatado y versionado

In [5]:
tag = df.columns[df.columns.str.match('^tag')]
param_cols = df.columns[df.columns.str.match('^param')].tolist()
metadata_model_version =  param_cols

## USE CASE 1 - THERAPY MODELLING

Recogemos cual ha sido nuestro mejor modelo seleccionado en base a la métrica `f1_samples` ( debido al debalance de clases ) entre todas las evaluaciones por combinatorias cohorte

In [6]:
df_multilabel = df[(df["experiment_id"]==1) & (df["param.model_selection_shap_skipped"]!="TRUE")]

metrics_therapy = df_multilabel.columns[(df_multilabel.columns.str.match('.*therapy.*')) | ( df_multilabel.columns.str.match('.*surgery.*'))].tolist()
metrics_therapy =  metrics_therapy + ['metric.test_f1_macro','metric.test_f1_samples','metric.test_subset_accuracy']

print("Comparacion con f1 ponderada")
mean_scores_f1_samples = df_multilabel.groupby(['param.selected_model'])['metric.test_f1_samples'].mean().idxmax
print(mean_scores_f1_samples)

print("\nComparacion con f1 global")
mean_scores_f1_macro = df_multilabel.groupby(['param.selected_model'])['metric.test_f1_macro'].mean().idxmax
print(mean_scores_f1_macro)

print("\nComparacion con acc")
mean_scores_acc = df_multilabel.groupby(['param.selected_model'])['metric.test_subset_accuracy'].mean().idxmax
print(mean_scores_acc)

Comparacion con f1 ponderada
<bound method Series.idxmax of param.selected_model
random_forest    0.676429
xgboost          0.643378
Name: metric.test_f1_samples, dtype: float64>

Comparacion con f1 global
<bound method Series.idxmax of param.selected_model
random_forest    0.673385
xgboost          0.682073
Name: metric.test_f1_macro, dtype: float64>

Comparacion con acc
<bound method Series.idxmax of param.selected_model
random_forest    0.231432
xgboost          0.193184
Name: metric.test_subset_accuracy, dtype: float64>


RF trata mejor clases minoritarias o balance

XGB optimiza rendimiento global

Ya que el f1 global no es considerablemente mayor en XGBOOST y queremos que se ajuste lo mejor a las clases minoritarias el modelo, elegimos el Random Forest.

En cuanto a precisión global, es demasiado bajo para utilizarlo en la práctica

Vemos con más detalle la precisión por clase y cohorte en Random Forest

In [7]:
eval_cdu_1 = (df_multilabel[metadata_model_version+metrics_therapy]
.drop(['param.inner_cv_splits','param.inner_cv_strategy','param.model_selection_shap_skipped'], axis=1)
.drop_duplicates()).dropna()

In [16]:
# eval_cdu_1.to_csv('./usecase_1_pertarget_metrics.tsv', sep='\t', index =False)

In [8]:
eval_cdu_1

,param.held_out_cohort,param.selected_model,metric.breast_surgery_BREAST_CONSERVING_accuracy,metric.breast_surgery_BREAST_CONSERVING_f1_binary,metric.breast_surgery_MASTECTOMY_accuracy,metric.breast_surgery_MASTECTOMY_f1_binary,metric.chemotherapy_accuracy,metric.chemotherapy_f1_binary,metric.hormone_therapy_accuracy,metric.hormone_therapy_f1_binary,metric.radio_therapy_accuracy,metric.radio_therapy_f1_binary,metric.test_f1_macro,metric.test_f1_samples,metric.test_subset_accuracy
30,5.0,xgboost,0.653465,0.477612,0.643564,0.731343,0.950495,0.915254,0.683168,0.636364,0.415842,0.535433,0.659201,0.602876,0.207921
31,4.0,xgboost,0.632768,0.551724,0.655367,0.670270,0.909605,0.804878,0.666667,0.742358,0.587571,0.724528,0.698752,0.676110,0.158192
32,3.0,xgboost,0.590909,0.439024,0.592885,0.677116,0.956522,0.887755,0.679842,0.716783,0.590909,0.720648,0.688265,0.651148,0.213439
33,2.0,random_forest,0.615385,0.540816,0.628205,0.639004,0.987179,0.969072,0.722222,0.765343,0.717949,0.833333,0.749514,0.710277,0.307692
34,1.0,random_forest,0.665025,0.517730,0.667488,0.732673,0.640394,0.223404,0.672414,0.743738,0.642857,0.768740,0.597257,0.642582,0.155172


## USE CASE 2 - SURVIVAL MODELLING

Recogemos cual ha sido nuestro mejor modelo seleccionado en base a la métrica `r2` ( debido al debalance de clases ) entre todas las evaluaciones por combinatorias cohorte

In [13]:
df_regression = df[df["experiment_id"]==2]

metrics_prog =   ['metric.test_rmse','metric.test_mae','metric.test_r2']

print("Comparacion con R2")
mean_scores_f1_samples = df_regression.groupby(['param.selected_model'])['metric.test_r2'].mean().idxmax
print(mean_scores_f1_samples)

print("\nComparacion con MAE")
mean_scores_f1_macro = df_regression.groupby(['param.selected_model'])['metric.test_mae'].mean().idxmax
print(mean_scores_f1_macro)

print("\nComparacion con RMSE")
mean_scores_acc = df_regression.groupby(['param.selected_model'])['metric.test_rmse'].mean().idxmax
print(mean_scores_acc)

Comparacion con R2
<bound method Series.idxmax of param.selected_model
random_forest    0.136558
Name: metric.test_r2, dtype: float64>

Comparacion con MAE
<bound method Series.idxmax of param.selected_model
random_forest    47.204016
Name: metric.test_mae, dtype: float64>

Comparacion con RMSE
<bound method Series.idxmax of param.selected_model
random_forest    59.360406
Name: metric.test_rmse, dtype: float64>


In [14]:
eval_cdu_2 = (df_regression[metadata_model_version+metrics_prog]
.drop(['param.inner_cv_splits','param.inner_cv_strategy','param.model_selection_shap_skipped'], axis=1)
.drop_duplicates()).dropna()

In [18]:
# eval_cdu_2.to_csv('./usecase_2_pertarget_metrics.tsv', sep='\t', index =False)

In [15]:
eval_cdu_2

,param.held_out_cohort,param.selected_model,metric.test_rmse,metric.test_mae,metric.test_r2
40,5.0,random_forest,64.100901,51.945777,0.110176
41,4.0,random_forest,45.904809,37.908712,0.378327
42,3.0,random_forest,69.550108,52.471586,0.106225
43,2.0,random_forest,68.388846,52.469425,0.239997
44,1.0,random_forest,48.857366,41.224582,-0.151935
51,5.0,random_forest,64.100901,51.945777,0.110176
54,2.0,random_forest,68.388846,52.469425,0.239997


Los resultados no indican capacidad predictiva limitada, puesto que el único modelo que ha superado el entrenamiento de regresion es  explica solo 13.7% de la varianza.

## USE CASE 3 - DIAGNOSIS MODELLING

Recogemos cual ha sido nuestro mejor modelo seleccionado en base a la métrica `f1_weighted`

In [19]:
df_multiclass = df[(df["experiment_id"]==3)&(~df["tag.inner_strategy"].isna())]

metrics_diagnosis = df_multiclass.columns[(df_multiclass.columns.str.match('.*ctree.*'))].tolist()
metrics_diagnosis =  metrics_diagnosis + ['metric.test_subset_accuracy']

print("Comparacion con precision")
mean_scores_f1_samples = df_multiclass.groupby(['param.selected_model'])['metric.test_exact_match_accuracy'].mean().idxmax
print(mean_scores_f1_samples)


Comparacion con precision
<bound method Series.idxmax of param.selected_model
xgboost    0.989347
Name: metric.test_exact_match_accuracy, dtype: float64>


In [20]:
eval_cdu_3 = (df_multiclass[metadata_model_version+metrics_diagnosis]
.drop(['param.inner_cv_splits','param.inner_cv_strategy','param.model_selection_shap_skipped'], axis=1)
.drop_duplicates()[0:5])
eval_cdu_3

,param.held_out_cohort,param.selected_model,metric.orctree_breast_accuracy,metric.orctree_breast_f1_weighted,metric.orctree_idc_accuracy,metric.orctree_idc_f1_weighted,metric.orctree_ilc_accuracy,metric.orctree_ilc_f1_weighted,metric.orctree_immc_accuracy,metric.orctree_immc_f1_weighted,metric.orctree_mbc_accuracy,metric.orctree_mbc_f1_weighted,metric.orctree_mdlc_accuracy,metric.orctree_mdlc_f1_weighted,metric.orctree_nan_accuracy,metric.orctree_nan_f1_weighted,metric.test_subset_accuracy
0,5.0,xgboost,1.0,1.0,1.000000,1.000000,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,1.0,1.000000,1.000000,NaN
1,4.0,xgboost,1.0,1.0,0.995763,0.995788,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,1.0,0.978814,0.968334,NaN
2,3.0,xgboost,1.0,1.0,0.991826,0.991865,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,1.0,0.990463,0.985718,NaN
3,2.0,xgboost,1.0,1.0,0.993007,0.993039,1.0,1.0,1.0,1.0,0.996503,0.994758,1.0,1.0,0.996503,0.998249,NaN
4,1.0,xgboost,1.0,1.0,0.993763,0.993774,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,1.0,0.991684,0.990307,NaN


In [21]:
eval_cdu_3.to_csv('./usecase_3_pertarget_metrics.tsv', sep='\t', index =False)


en este caso la estrategia de uso de esta serie de modelos ha sido correcta, ya que se traduce a todas las combinaciones de cohortes con buenas métricas